# Full Baseline Pipeline

## 1. Install

In [ ]:
!pip install -q timm albumentations

## 2. Imports

In [ ]:
import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm.auto import tqdm


## 3. Config

In [ ]:
class CFG:

    num_workers=2
    
    img_size = 256
    batch_size = 64
    epochs = 10

    lr = 1.5e-4
    weight_decay = 1e-4

    val_size = 0.1
    num_classes = 8

    device = "cuda"

    # soft-voting: resnet34 + resnet50 teachers + lenet5 student
    ensemble_weights = [0.4, 0.4, 0.2]

    image_dir = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/images"

    train_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/train.csv"
    add_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/additional_train.csv"
    test_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/test.csv"

    # set these checkpoints after training each model
    ckpt_resnet34 = "/kaggle/working/model_best_resnet34.pth"
    ckpt_resnet50 = "/kaggle/working/model_best_resnet50.pth"
    ckpt_lenet5 = "/kaggle/working/model_student_lenet5.pth"


## 4A. Load train dataset

In [ ]:
train = pd.read_csv(CFG.train_csv)
add = pd.read_csv(CFG.add_csv)

df = pd.concat([train, add], ignore_index=True)

# remove unlabeled samples if present
df = df[df.pen_id != -1].reset_index(drop=True)

print("Training samples:", len(df))
print(df.head())

## 4B. Load test dataset

In [ ]:
test_df = pd.read_csv(CFG.test_csv)

print("Test samples:", len(test_df))
print(test_df.head())

## 4C. Assign full image_path

In [ ]:
df["image_path"] = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/" + df["image_path"]
test_df["image_path"] = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/" + test_df["image_path"]

## 4D. debug 1

In [ ]:
df["pen_id"] = df["pen_id"] - 1

## 5A. Sharpness-Focused Augmentation

In [ ]:
train_tfms = A.Compose([

    A.RandomResizedCrop(
        size=(CFG.img_size, CFG.img_size),
        scale=(0.85, 1.0),
        ratio=(0.95, 1.05),
        p=1
    ),

    # small rotation (drawing angle)
    A.Affine(
        rotate=(-10, 10),
        translate_percent=(0.02, 0.02),
        scale=(0.95, 1.05),
        p=0.7
    ),

    A.HorizontalFlip(p=0.5),

    # enhance circle stroke
    A.CLAHE(clip_limit=2.0, p=0.5),

    # sharpen stroke edges
    A.Sharpen(alpha=(0.2, 0.6), lightness=(0.9, 1.2), p=0.5),

    # simulate scanner brightness
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.2,
        p=0.5
    ),

    # ink variation
    A.RandomGamma(
        gamma_limit=(80,120),
        p=0.4
    ),

    # subtle blur
    A.GaussianBlur(blur_limit=(3,5), p=0.15),

    # stroke deformation
    A.ElasticTransform(
        alpha=8,
        sigma=6,
        p=0.25
    ),

    # noise from scanning
    A.GaussNoise(var_limit=(5,20), p=0.3),

])
valid_tfms = A.Compose([

    A.Resize(CFG.img_size, CFG.img_size),

])

## 5B. Removed extra preprocessing for 3-layer RGB only


In [ ]:
# removed: ring crop for simpler 3-channel training


## 5C. Removed extra preprocessing for 3-layer RGB only


In [ ]:
# removed: laplacian texture channel


## 5D. Removed extra preprocessing for 3-layer RGB only


In [ ]:
# removed: ink density channel


## 6. Dataset (train)

In [ ]:
class CircleDataset(Dataset):

    def __init__(self, df, transforms=None):

        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # image_path is already full path
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # apply augmentation on RGB image only
        if self.transforms:
            img = self.transforms(image=img)["image"]

        # normalize
        img = img.astype(np.float32) / 255.0

        img = torch.from_numpy(img).permute(2, 0, 1)

        label = row.pen_id

        return img, label


## 7. MixUp / CutMix

In [ ]:
def mixup_data(x, y, alpha=0.4):

    lam = np.random.beta(alpha, alpha)

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()

    mixed_x = lam * x + (1 - lam) * x[index]

    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

## 8. Model

In [ ]:
class TimmPenClassifier(nn.Module):

    def __init__(self, backbone_name):
        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )

        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.4),
            nn.Linear(in_features, CFG.num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

class ResNet34Pen(TimmPenClassifier):
    def __init__(self):
        super().__init__("resnet34")

class ResNet50Pen(TimmPenClassifier):
    def __init__(self):
        super().__init__("resnet50")

class LeNet5Student(nn.Module):
    def __init__(self, num_classes=CFG.num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        # keep lenet-style classifier by normalizing feature map size
        self.pool = nn.AdaptiveAvgPool2d((5, 5))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(inplace=True),
            nn.Linear(120, 84),
            nn.ReLU(inplace=True),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


## 9. SWA Setup

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR

## 10. Training Loop

In [ ]:
def train_fn(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    for x, y in pbar:

        x, y = x.cuda(), y.cuda()

        if random.random() < 0.5:
            x, ya, yb, lam = mixup_data(x, y)

            pred = model(x)
            loss = lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

        else:
            pred = model(x)
            loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # show running loss
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

## 11. Validation

In [ ]:
def valid_fn(model, loader, criterion):

    model.eval()

    preds = []
    targets = []
    total_loss = 0

    pbar = tqdm(loader, desc="Valid", leave=False)

    with torch.no_grad():
        for x, y in pbar:

            x, y = x.cuda(), y.cuda()

            pred = model(x)

            loss = criterion(pred, y)
            total_loss += loss.item()

            preds.append(pred.softmax(1).cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    acc = (preds.argmax(1) == targets).mean()

    val_loss = total_loss / len(loader)

    return val_loss, acc

## 12A. Single Split Training (No Fold)


In [ ]:
print(df.pen_id.min(), df.pen_id.max())

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=CFG.val_size,
    random_state=42,
    stratify=df.pen_id
)

train_ds = CircleDataset(train_df, train_tfms)
val_ds = CircleDataset(val_df, valid_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    drop_last=True,
    pin_memory=True
)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size)

model = ResNet34Pen().cuda()

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
criterion = nn.CrossEntropyLoss()

swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=3e-4, anneal_strategy="cos")

best_acc = 0
best_swa_acc = 0

for epoch in range(CFG.epochs):

    train_loss = train_fn(model, train_loader, optimizer, criterion)

    val_loss, acc = valid_fn(model, val_loader, criterion)

    print(
        f"Epoch {epoch+1}/{CFG.epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={acc:.4f}"
    )

    # save best normal model
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "model_best.pth")
        print("✅ Saved BEST normal model")

    # SWA phase
    if epoch > CFG.epochs * 0.75:

        swa_model.update_parameters(model)
        swa_scheduler.step()

        # evaluate SWA model
        swa_loss, swa_acc = valid_fn(swa_model, val_loader, criterion)

        print(f"SWA_acc={swa_acc:.4f}")

        if swa_acc > best_swa_acc:
            best_swa_acc = swa_acc
            torch.save(swa_model.state_dict(), "model_best_swa.pth")
            print("🔥 Saved BEST SWA model")

## 12B. Load best models

In [ ]:
def _extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for key in ["state_dict", "model", "model_state_dict", "net", "weights"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                return ckpt[key]
    return ckpt

def _adapt_state_dict_for_model(state_dict, model):
    model_keys = set(model.state_dict().keys())
    sd = state_dict

    # remove wrappers like module.* / model.*
    for prefix in ["module.", "model."]:
        if sd and all(k.startswith(prefix) for k in sd):
            sd = {k[len(prefix):]: v for k, v in sd.items()}

    # remap plain torchvision resnet keys -> timm backbone keys
    if any(k.startswith(("conv1.", "bn1.", "layer")) for k in sd) and not any(k.startswith("backbone.") for k in sd):
        sd = {((f"backbone.{k}") if k.startswith(("conv1.", "bn1.", "layer")) else k): v for k, v in sd.items()}

    # remap fc -> custom head classifier when needed
    if "fc.weight" in sd and "head.2.weight" in model_keys and "head.2.weight" not in sd:
        sd["head.2.weight"] = sd.pop("fc.weight")
    if "fc.bias" in sd and "head.2.bias" in model_keys and "head.2.bias" not in sd:
        sd["head.2.bias"] = sd.pop("fc.bias")

    return sd

def load_checkpoint_flexible(model, ckpt_path):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = _extract_state_dict(ckpt)
    state_dict = _adapt_state_dict_for_model(state_dict, model)

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"[warn] {ckpt_path}: missing keys ({len(missing)}), e.g. {missing[:5]}")
    if unexpected:
        print(f"[warn] {ckpt_path}: unexpected keys ({len(unexpected)}), e.g. {unexpected[:5]}")
    return model

def build_ensemble_models():
    model34 = ResNet34Pen()
    load_checkpoint_flexible(model34, CFG.ckpt_resnet34)
    model34.cuda()

    model50 = ResNet50Pen()
    load_checkpoint_flexible(model50, CFG.ckpt_resnet50)
    model50.cuda()

    student = LeNet5Student()
    load_checkpoint_flexible(student, CFG.ckpt_lenet5)
    student.cuda()

    return [model34, model50, student]

models = build_ensemble_models()


## 13. Test Dataset

In [ ]:
class TestDataset(Dataset):

    def __init__(self, df, transforms=None, image_root="/kaggle/input/competitions/icdar-2026-circleid-pen-classification/images"):
        self.df = df.reset_index(drop=True).copy()
        self.transforms = transforms
        self.image_root = image_root

        # Prevent common inference crash when sample_submission has only image_id.
        if "image_path" not in self.df.columns:
            if "image_id" not in self.df.columns:
                raise ValueError("TestDataset expects either 'image_path' or 'image_id' column.")
            self.df["image_path"] = self.df["image_id"].astype(str).apply(
                lambda x: f"{self.image_root}/{x}.png"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = cv2.imread(row["image_path"])
        if img is None:
            raise FileNotFoundError(f"Could not read image: {row['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ensure fixed size (example: 224x224)
        img = cv2.resize(img, (224, 224))

        if self.transforms:
            img = self.transforms(image=img)["image"]

        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2,0,1)

        return img


## 14. TTA Inference

In [ ]:
def inference(models, loader, model_weights=None):

    if model_weights is None:
        model_weights = [1.0 / len(models)] * len(models)

    if len(model_weights) != len(models):
        raise ValueError("len(model_weights) must match len(models)")

    weights = np.asarray(model_weights, dtype=np.float32)
    if np.any(weights < 0):
        raise ValueError("model_weights must be non-negative")
    if weights.sum() <= 0:
        raise ValueError("sum(model_weights) must be > 0")
    weights = weights / weights.sum()

    preds = []

    for model in models:
        model.eval()
        fold_preds = []
        with torch.no_grad():
            for x in loader:
                x = x.cuda(non_blocking=True)
                p = model(x).softmax(1)
                p2 = model(torch.flip(x, [3])).softmax(1)
                p = (p + p2) / 2
                fold_preds.append(p.cpu().numpy())
        preds.append(np.concatenate(fold_preds, axis=0))

    preds = np.stack(preds, axis=0)
    return np.tensordot(weights, preds, axes=(0, 0))


## 15. Submission

In [ ]:
# Load sample submission (correct order for Kaggle)
test_df = pd.read_csv(
    "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/sample_submission.csv"
)

# Build full image path
test_df["image_path"] = (
    "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/images/"
    + test_df["image_id"].astype(str)
    + ".png"
)

# Create dataset
test_ds = TestDataset(test_df)

# DataLoader
test_loader = DataLoader(
    test_ds,
    batch_size=32,
    num_workers=2,
    pin_memory=True
)

# Inference
preds = inference(models, test_loader, model_weights=CFG.ensemble_weights)

# Convert 0..7 → 1..8
test_df["pen_id"] = preds.argmax(1) + 1

# Save submission
test_df[["image_id", "pen_id"]].to_csv("submission.csv", index=False)